# Step 3c: Retrain bge-base with Hard Negatives

**Goal:** Retrain `bge-base-en-v1.5` using mined hard negatives instead of random in-batch negatives.

**What changes vs previous runs:**
- Loss: `TripletLoss` instead of `MultipleNegativesRankingLoss`
- Data: `(query, positive, hard_negative)` triplets from `hard_negative_pairs.json`
- Base model: `bge-base-en-v1.5` (same as step 2 — fair comparison)
- Everything else identical: 2700 pairs, 5 epochs, batch 32, warmup 100

**Output:** `experiments/bge_hardneg/model` + FAISS index

## 0. Runtime check

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Not on T4. Go to Runtime > Change runtime type > T4 GPU')
else:
    print('Good to go.')

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers faiss-cpu accelerate

## 2. Upload files

You need to upload two files from your Mac:
- `data/hard_negative_pairs.json` — the mined triplets
- `experiments/baseline/baseline_chunks.pkl` — for building the FAISS index

Run the cell below and select both files when the dialog appears.

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/experiments/baseline', exist_ok=True)
os.makedirs('/content/experiments/bge_hardneg/model', exist_ok=True)

print('Upload hard_negative_pairs.json and baseline_chunks.pkl')
uploaded = files.upload()

for fname, data in uploaded.items():
    if 'hard_negative' in fname:
        dest = '/content/data/hard_negative_pairs.json'
    elif 'chunks' in fname:
        dest = '/content/experiments/baseline/baseline_chunks.pkl'
    else:
        dest = f'/content/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved {fname} -> {dest}')

## 3. Config

In [ ]:
import random
import numpy as np

BASE_MODEL    = 'BAAI/bge-base-en-v1.5'
EPOCHS        = 5
BATCH_SIZE    = 32
WARMUP_STEPS  = 100
SEED          = 42
OUT_DIR       = '/content/experiments/bge_hardneg/model'

DATA_DIR        = '/content/data'
EXPERIMENTS_DIR = '/content/experiments'

random.seed(SEED)
np.random.seed(SEED)

print(f'Base model : {BASE_MODEL}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Output     : {OUT_DIR}')

## 4. Load triplets

In [ ]:
import json

with open('/content/data/hard_negative_pairs.json') as f:
    all_triplets = json.load(f)

# Same split logic as all previous runs
# hard_negative_pairs.json already contains exactly 2699 training triplets
# (mined from the 2700 train pairs; 1 was skipped during mining)
# Use first 2499 for training, last 200 for validation
VAL_SIZE      = 200
train_triplets = all_triplets[:-VAL_SIZE]
val_triplets   = all_triplets[-VAL_SIZE:]

print(f'Train triplets : {len(train_triplets)}')
print(f'Val triplets   : {len(val_triplets)}')
print(f'\nExample:')
ex = train_triplets[0]
print(f'  Query    : {ex["query"][:80]}')
print(f'  Positive : {ex["positive"][:80]}')
print(f'  Hard neg : {ex["hard_negative"][:80]}')

## 5. Build validation evaluator

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries  = {str(i): t['query']    for i, t in enumerate(val_triplets)}
corpus   = {str(i): t['positive'] for i, t in enumerate(val_triplets)}
relevant = {str(i): {str(i)}      for i in range(len(val_triplets))}

evaluator = InformationRetrievalEvaluator(
    queries, corpus, relevant, name='val-hardneg'
)
print(f'Validation evaluator ready — {len(val_triplets)} triplets')

## 6. Train with TripletLoss (~10 min on T4)

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import TripletLoss
from torch.utils.data import DataLoader

# TripletLoss expects: anchor, positive, negative
examples = [
    InputExample(texts=[t['query'], t['positive'], t['hard_negative']])
    for t in train_triplets
]

model  = SentenceTransformer(BASE_MODEL)
loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss   = TripletLoss(model)

total_steps = len(loader) * EPOCHS
print(f'Training — {len(examples)} triplets, {EPOCHS} epochs, {total_steps} steps')

model.fit(
    train_objectives=[(loader, loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluation_steps=len(loader),
    output_path=OUT_DIR,
    save_best_model=True,
    show_progress_bar=True
)

print(f'\nSaved to {OUT_DIR}')

## 7. Build FAISS index

In [ ]:
import pickle
import faiss

with open('/content/experiments/baseline/baseline_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} chunks')

model      = SentenceTransformer(OUT_DIR)
texts      = [c['text'] for c in chunks]
embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

idx_path = '/content/experiments/bge_hardneg/bge_hardneg.faiss'
faiss.write_index(index, idx_path)
print(f'Saved index: {index.ntotal} vectors, dim={dim}')

## 8. Sanity check — Hit@3 on held-out synthetic (n=100)

In [ ]:
# Load held-out eval pairs from the original training_pairs.json
# We don't have that file here, so we use the last 100 of the mined triplets
# as a proxy (same queries, different format)
# NOTE: proper eval runs on your M2 with step4d_eval_bge.py

held_out = all_triplets[-100:]
hits = 0
model = SentenceTransformer(OUT_DIR)
index = faiss.read_index(idx_path)

for t in held_out:
    q_emb = model.encode([t['query']], normalize_embeddings=True)
    _, ids = index.search(q_emb, 3)
    retrieved = [chunks[i]['text'] for i in ids[0] if i < len(chunks)]
    if any(t['positive'][:100] in r for r in retrieved):
        hits += 1

print(f'Sanity Hit@3 (proxy, n=100): {hits/100:.4f}')
print('Run step4d_eval_bge.py on your Mac for proper eval with CIs.')

## 9. Download model + index

In [ ]:
import shutil
from google.colab import files

# Zip the bge_hardneg experiment folder
shutil.make_archive('/content/bge_hardneg', 'zip', '/content/experiments', 'bge_hardneg')
files.download('/content/bge_hardneg.zip')
print('Download started. Unzip into your experiments/ folder on Mac.')
print('Then run: python step4d_eval_bge.py --eval manual')